<a href="https://colab.research.google.com/github/gyeongee/TIL/blob/main/%EB%A1%9C%EC%A7%80%EC%8A%A4%ED%8B%B1%ED%9A%8C%EA%B7%80_%EC%9C%84%EC%8A%A4%EC%BD%98%EC%8B%A0%EC%9C%A0%EB%B0%A9%EC%95%94%EB%8D%B0%EC%9D%B4%ED%84%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

cancer = load_breast_cancer()

In [3]:
# 정규화 진행
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# StandardScaler()로 평균이 0, 분산 1로 데이터 분포도 변환
scaler=StandardScaler()
data_scaled=scaler.fit_transform(cancer.data)

X_train,X_test,y_train,y_test = train_test_split(data_scaled,cancer.target,test_size=0.3,random_state=0)



In [6]:
# 로지스틱 회귀로 학습 및 예측 수행, 정확도와 ROC-AUC
from sklearn.metrics import accuracy_score,roc_auc_score

lr_clf = LogisticRegression()
lr_clf.fit(X_train,y_train)
lr_preds=lr_clf.predict(X_test)

# accuracy와 roc-auc 측정
print(f"accuracy:{accuracy_score(y_test,lr_preds):.3f}, roc_auc:{roc_auc_score(y_test,lr_preds):.3f}")

accuracy:0.977, roc_auc:0.972


In [9]:
# 여러 개의 solver 값별로 LogisticRegression 학습 후 성능 평가
solvers={'lbfgs','liblinear','newton-cg','sag','saga'}

for solver in solvers:
  lr_clf=LogisticRegression(solver=solver, max_iter=600)
  lr_clf.fit(X_train,y_train)
  lr_preds=lr_clf.predict(X_test)
  print(f"solver:{solver}, accuracy:{accuracy_score(y_test,lr_preds):.3f}, roc_aud:{roc_auc_score(y_test,lr_preds):.3f}")


solver:saga, accuracy:0.982, roc_aud:0.979
solver:newton-cg, accuracy:0.977, roc_aud:0.972
solver:liblinear, accuracy:0.982, roc_aud:0.979
solver:lbfgs, accuracy:0.977, roc_aud:0.972
solver:sag, accuracy:0.982, roc_aud:0.979


In [12]:
# GridSearchCV를 이용해 위스콘신 데이터 세트에서 solver,penalty,C 최적화
from sklearn.model_selection import GridSearchCV

params={
    'solver':['liblinear','lbfgs'],
    'penalty':['l2','l1'],
    'C':[0.01,0.1,1,5,10]
}

lr_clf =LogisticRegression()

grid_clf = GridSearchCV(lr_clf,param_grid=params,scoring='accuracy',cv=3)
grid_clf.fit(data_scaled,cancer.target)
print(f"최적의 하이퍼 파라미터:{grid_clf.best_params_}, 최적 평균 정확도:{grid_clf.best_score_}")

최적의 하이퍼 파라미터:{'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}, 최적 평균 정확도:0.9789102385593614


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
15 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py", line 1193, in fit
    solver = _check_solver

solver가 liblinear,Penalty가 L2 규제, C값은 0.1일 때, 평균 정확도가 0.979로 가장 좋은 성능을 나타내었습니다.